### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="ghanas_indigenous_intel",
    dataset_year="2025",
    domain_str="environmental science & climate",
    # Data Source
    dataset_source="Zindi",
    original_dataset_source_download_link="https://www.kaggle.com/datasets/phantom50/ghanas-indigenous-intel-dataset",
    download_description=r"""
We download the data from Kaggle, although it was originally a Zindi competition.

kaggle datasets download phantom50/ghanas-indigenous-intel-dataset \
&& unzip ghanas-indigenous-intel-dataset.zip \
&& rm ghanas-indigenous-intel-dataset.zip \
&& mkdir -p local-data-warehouse/ghanas_indigenous_intel \
&& mv *.csv local-data-warehouse/ghanas_indigenous_intel/
""",
    # References
    academic_reference_bibtex=r"""@misc{zindi_ghana_indigenous_intel_2025,
    author       = {{Zindi}},
    title        = {Ghana's Indigenous Intel Challenge [BEGINNERS ONLY]: Data},
    year         = {2025},
    howpublished = {\url{https://zindi.africa/competitions/ghana-indigenous-intel-challenge/data}},
    note         = {Zindi dataset page. Accessed 2026-04-11}
}
""",
    academic_reference_bibtex_key="zindi_ghana_indigenous_intel_2025",
    license="CC-BY-SA 4.0",
    data_tags=["Non-IID", "Temporal", "Grouped"],
    curation_comments="""
- Information about the data source: The data was collected by local farmers in the Pra River Basin of Ghana using the Smart Indigenous Weather App, a custom-built mobile app for data collection. Farmers made weather forecasts based on traditional signs such as cloud formations, sun position, wind, moon, heat, and specific tree or animal behaviors. The task is to predict the farmers' forecasts based on these traditional signs, which are represented as features in the dataset. The target variable is the actual amount of rain in four categories, therefore the actual task is to make a prediction correcting the farmers' forecasts.
- The data was initially used in a Zindi competition with a temporal split, we do the same.
- We create 6 splits using a sliding window approach with a window size of 3 days and a step size of 2 days. This means that one day overlaps among consecutive test splits.
- We ensure that each split uses at least half of the available samples for training.
- We drop the ID column.
- We convert prediction_time to datetime and sort by it.
- We rename the target column to "rainfall" for better clarity.
- We transform object columns and user_id to categorical.

- Note: The split that would be ideal for the task would be grouped+temporal. However, the available data is not enough for this scenario. In general the available data size reduces the task quality a lot.
- Anomaly: For this dataset it is possible that test folds don't contain some classes.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="rainfall",
    problem_type="multiclass_classification",
    objective_metric_name="f1_macro",
    stratify_on="rainfall",
    time_on="prediction_time",
)

## Preprocessing

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(dataset_mold.path / "train.csv")
test_df = pd.read_csv(dataset_mold.path / "test.csv")

df = df.drop(columns=["ID"])
test_df = test_df.drop(columns=["ID"])

df["prediction_time"] = pd.to_datetime(df["prediction_time"])
test_df["prediction_time"] = pd.to_datetime(test_df["prediction_time"])

df = df.rename(columns={"Target": "rainfall"})

cat_cols = ['user_id', 'community', 'district', 'indicator', 'indicator_description', 'time_observed', 'rainfall']
for col in cat_cols:
    df[col] = df[col].astype("category")

print("Loaded data shape:", df.shape)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    target_feature=task_mold.target_column_name,
    problem_type=task_mold.problem_type,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

target_col = task_mold.target_column_name

day_of_year = df["prediction_time"].dt.dayofyear

window = 3
step = 2

max_day = day_of_year.max()
last_start = max_day - window + 1
min_train_size = len(df) / 2

# walk backward from the last valid split start
valid_starts = []
split = last_start

while split >= day_of_year.min():
    train_size = (day_of_year < split).sum()
    if train_size < min_train_size:
        break
    valid_starts.append(split)
    split -= step

valid_starts = sorted(valid_starts)

used_in_train = set()
used_in_test = set()
used_data = set()
splits = {}
for s_num, split in enumerate(valid_starts[::-1]):
    train_idx = day_of_year[day_of_year < split].index
    test_idx = day_of_year[(day_of_year >= split) & (day_of_year < split + window)].index

    train_size = (day_of_year < split).sum()
    test_size = ((day_of_year >= split) & (day_of_year < split + window)).sum()

    splits[s_num] = {0: [train_idx.tolist(), test_idx.tolist()]}

    used_in_train.update(train_idx)
    used_in_test.update(test_idx)
    used_data.update(train_idx)
    used_data.update(test_idx)

    print(
        f"Train size (days < {split}) = {train_size}, "
        f"Test size (days {split}-{split + window - 1}) = {test_size}"
    )
    print("Train target dist:", df.loc[train_idx, target_col].value_counts(normalize=True))
    print("Test target dist:", df.loc[test_idx, target_col].value_counts(normalize=True))

    assert len(set(train_idx).intersection(set(test_idx)))==0, "Train and test indices overlap!"

print(f"{len(splits)} splits created.")
print(f"{len(used_data)/df.shape[0]:.4f} of the samples are used.")
print(f"{len(used_in_train)/df.shape[0]:.4f} of the samples are used in training")
print(f"{len(used_in_test)/df.shape[0]:.4f} of the samples are used in testing.")

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="We create 6 splits using a sliding window approach with a window size of 3 days and a step size of 2 days. This means that one day overlaps among consecutive test splits. We ensure that each split uses at least half of the available samples for training. ",
    splits=splits,
    time_horizon=3,
    time_horizon_unit="days",
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()